# 🎙️ Pipeline Kafka de Análisis Vocal en Streaming

## Idea B — Integración de todos los trabajos

**Integrantes:** Balda Javier · Caracoix Juan · Casas Facundo  
**Institución:** Universidad Católica Argentina  
**Materia:** Análisis y Procesamiento de Datos Streaming

---

## 📌 Arquitectura del sistema

```
┌──────────────────────────────────────────────────────────────────────────────────┐
│  FUENTES DE DATOS                                                                │
│  realtime_frames.csv (pipeline.py pYIN)  ←  challenge_streaming original        │
│  artist_profiles.py (6 perfiles)         ←  vocal_comparador (Idea C)           │
└──────────────────────┬───────────────────────────────────────────────────────────┘
                       │ vocal_producer.py
                       │ 1 frame / 50ms  (20fps)
                       ▼
┌──────────────────────────────────────────────────────────────────────────────────┐
│  KAFKA KRaft  (broker + controller, sin Zookeeper)                               │
│                                                                                  │
│   vocal.frames   (partitions=1)   ← frames crudos del audio                     │
│   vocal.analyzed (partitions=1)   ← predicción de tipo vocal por ventana         │
│   vocal.alerts   (partitions=1)   ← anomalías vocales detectadas                 │
└──────────────────────┬───────────────────────────────────────────────────────────┘
                       │ vocal_consumer.py
                       ▼
┌────────────────────────────────────────────────────────┐
│  CONSUMER (buffer deslizante 2s / stride 0.5s)         │
│                                                        │
│  extract_features()  ──► VocalClassifier (Idea A)      │
│       ↓ cada 10 frames                                 │
│  vocal.analyzed: {prediction, confidence, probas}      │
│                                                        │
│  VocalAnomalyDetector (Idea D) ─► vocal.alerts         │
│  {BREAK_VOZ, CLIMAX_VOCAL, CAIDA_INTENS, ...}          │
└───────────────────────────┬────────────────────────────┘
                            │
                            ▼
                  kafka_dashboard.html
              (Dashboard en vivo · Chart.js)
```

## Secciones

| # | Contenido | Tiempo estimado |
|---|-----------|----------------|
| 1 | Instalación: Java 17, Kafka KRaft, confluent-kafka | ~5 min |
| 2 | Configuración Kafka KRaft e inicio del broker | ~1 min |
| 3 | Crear tópicos (vocal.frames / vocal.analyzed / vocal.alerts) | < 1 min |
| 4 | Producer: emite frames desde CSV o modo demo | ~90s (realtime) |
| 5 | Consumer: clasificación + anomalías en tiempo real | paralelo al producer |
| 6 | Dashboard interactivo inline | < 1 min |
| 7 | Análisis de resultados del stream | < 1 min |

## Sección 1 · Instalación de dependencias

In [ ]:
# ── 1.1 Java 17 (requerido por Kafka) ────────────────────────────
import os, subprocess

print('📦 Instalando Java 17...')
subprocess.run(['apt-get','update','-qq'], capture_output=True)
subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jdk-headless'], capture_output=True)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

java = subprocess.run(['java','-version'], capture_output=True, text=True)
print('✅ Java:', java.stderr.split('\n')[0])

In [ ]:
# ── 1.2 Kafka KRaft (igual que en el trabajo de NLP) ─────────────
import subprocess, os, urllib.request

KAFKA_DIR = '/content/kafka'
VERSIONES = [('3.9.2','2.13'),('3.9.1','2.13'),('3.8.1','2.13')]

if not os.path.exists(KAFKA_DIR):
    descargado = False
    for version, scala in VERSIONES:
        tgz  = f'kafka_{scala}-{version}.tgz'
        url  = f'https://archive.apache.org/dist/kafka/{version}/{tgz}'
        dest = f'/tmp/{tgz}'
        print(f'⬇️  Intentando Kafka {version}...')
        try:
            r = subprocess.run(['wget','-q','--timeout=60','-O',dest,url],
                               capture_output=True)
            if r.returncode == 0 and os.path.getsize(dest) > 1_000_000:
                subprocess.run(['tar','-xzf',dest,'-C','/tmp'], capture_output=True)
                carpeta = f'/tmp/kafka_{scala}-{version}'
                if os.path.exists(carpeta):
                    os.rename(carpeta, KAFKA_DIR)
                    print(f'✅ Kafka {version} instalado en {KAFKA_DIR}')
                    descargado = True; break
        except Exception as e:
            print(f'  Error: {e}')
    if not descargado:
        raise RuntimeError('No se pudo descargar Kafka.')
else:
    print('✅ Kafka ya instalado.')

In [ ]:
# ── 1.3 confluent-kafka + dependencias del pipeline ───────────────
# Mismo cliente que usamos en el trabajo de NLP
!pip install confluent-kafka -q
!pip install numpy pandas scikit-learn -q
print('✅ Dependencias Python instaladas.')

## Sección 2 · Configuración e inicio de Kafka KRaft

In [ ]:
import subprocess, os, time

KAFKA_DIR   = '/content/kafka'
LOG_DIR     = '/tmp/kraft-vocal-logs'
CONFIG_PATH = f'{KAFKA_DIR}/config/kraft-vocal.properties'

# Configuración KRaft — igual que en el trabajo de NLP Yelp
kraft_config = """
process.roles=broker,controller
node.id=1
controller.quorum.voters=1@localhost:9093
listeners=PLAINTEXT://localhost:9092,CONTROLLER://localhost:9093
advertised.listeners=PLAINTEXT://localhost:9092
controller.listener.names=CONTROLLER
listener.security.protocol.map=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
log.dirs=/tmp/kraft-vocal-logs
num.network.threads=3
num.io.threads=8
socket.send.buffer.bytes=102400
socket.receive.buffer.bytes=102400
socket.request.max.bytes=104857600
num.partitions=1
num.recovery.threads.per.data.dir=1
offsets.topic.replication.factor=1
transaction.state.log.replication.factor=1
transaction.state.log.min.isr=1
default.replication.factor=1
auto.create.topics.enable=true
group.initial.rebalance.delay.ms=0
log.retention.hours=2
"""

with open(CONFIG_PATH, 'w') as f:
    f.write(kraft_config)
print('✅ Config KRaft escrita:', CONFIG_PATH)

In [ ]:
import subprocess, time

KAFKA_DIR   = '/content/kafka'
CONFIG_PATH = f'{KAFKA_DIR}/config/kraft-vocal.properties'

# Limpiar estado anterior
subprocess.run(['rm','-rf','/tmp/kraft-vocal-logs'], capture_output=True)
subprocess.run(['pkill','-f','kafka.Kafka'], capture_output=True)
time.sleep(1)

# Generar cluster ID
result = subprocess.run([f'{KAFKA_DIR}/bin/kafka-storage.sh','random-uuid'],
                        capture_output=True, text=True)
cluster_id = result.stdout.strip()
print(f'🔑 Cluster ID: {cluster_id}')

# Formatear storage
fmt = subprocess.run([f'{KAFKA_DIR}/bin/kafka-storage.sh','format',
                       '-t',cluster_id,'-c',CONFIG_PATH],
                     capture_output=True, text=True)
print('📂 Storage:', fmt.stdout.strip() or fmt.stderr.strip()[:80])

# Iniciar broker
subprocess.Popen(
    f'nohup {KAFKA_DIR}/bin/kafka-server-start.sh {CONFIG_PATH} '
    '> /tmp/kafka_vocal.log 2>&1 &',
    shell=True
)
print('🚀 Iniciando Kafka...', end='')

# Esperar disponibilidad
for i in range(20):
    time.sleep(2)
    r = subprocess.run(
        [f'{KAFKA_DIR}/bin/kafka-topics.sh','--bootstrap-server',
         'localhost:9092','--list'],
        capture_output=True, text=True, timeout=5
    )
    if r.returncode == 0:
        print(f' listo en {(i+1)*2}s')
        print('✅ Kafka KRaft activo en localhost:9092')
        break
    print('.', end='', flush=True)
else:
    print('\n⚠️  Timeout. Revisar /tmp/kafka_vocal.log')

## Sección 3 · Crear los 3 tópicos

In [ ]:
import subprocess

KAFKA_DIR = '/content/kafka'
BOOTSTRAP = 'localhost:9092'

TOPICS = [
    # (nombre,        particiones, descripción)
    ('vocal.frames',   1, 'Frames crudos del análisis pYIN (50ms/frame)'),
    ('vocal.analyzed', 1, 'Predicción de tipo vocal por ventana (2s)'),
    ('vocal.alerts',   1, 'Anomalías vocales detectadas en tiempo real'),
]

for topic, parts, desc in TOPICS:
    r = subprocess.run(
        [f'{KAFKA_DIR}/bin/kafka-topics.sh',
         '--bootstrap-server', BOOTSTRAP,
         '--create', '--if-not-exists',
         '--topic', topic,
         '--partitions', str(parts),
         '--replication-factor', '1'],
        capture_output=True, text=True
    )
    status = '✅' if r.returncode == 0 else '⚠️'
    print(f'{status} {topic:<22} → {desc}')

# Listar tópicos creados
r = subprocess.run([f'{KAFKA_DIR}/bin/kafka-topics.sh',
                    '--bootstrap-server', BOOTSTRAP, '--list'],
                   capture_output=True, text=True)
print('\nTópicos activos:', r.stdout.strip())

## Sección 4 · Producer: emitir frames vocales

In [ ]:
# Importar el módulo del producer
# (copiar vocal_producer.py al directorio de trabajo)
from vocal_producer import (
    conectar_producer, crear_topico,
    frames_desde_csv, frames_desde_perfil,
    producir, TOPIC, FRAME_INTERVAL_S
)
import os

print(f'Tópico de salida: {TOPIC}')
print(f'Intervalo entre frames: {FRAME_INTERVAL_S*1000:.0f}ms')

# Conectar
producer = conectar_producer()
print('\nListo para producir. Elegir fuente de datos en la siguiente celda.')

In [ ]:
# ── Opción A: desde el CSV del challenge (análisis real pYIN) ─────
CSV_PATH = 'results/realtime_frames.csv'  # ajustar ruta

if os.path.exists(CSV_PATH):
    print(f'✅ CSV encontrado: {CSV_PATH}')
    frames_source = frames_desde_csv(
        CSV_PATH,
        artist_id  = 'bogdan_sos',
        song_title = 'S.O.S — Bogdan Shuvalov',
    )
    fuente = 'CSV real (pYIN)'
else:
    print(f'⚠️  {CSV_PATH} no encontrado. Usando modo demo (soprano).')
    frames_source = frames_desde_perfil('soprano')
    fuente = 'demo (soprano sintética)'

print(f'Fuente seleccionada: {fuente}')

# ── Opción B: artista específico del comparador ───────────────────
# frames_source = frames_desde_perfil('baritono')  # o soprano, tenor_pop, etc.
# frames_source = frames_desde_perfil('contratenor')

In [ ]:
# ── Producir ──────────────────────────────────────────────────────
# NOTA: ejecutar en paralelo con la Sección 5 (consumer)
#       Abrir una segunda terminal y correr: python vocal_consumer.py
#       O usar threading (ver celda siguiente)

print('🚀 Iniciando producer...')
print('   Tip: ejecutar la Sección 5 en otra celda mientras este corre.\n')

n_emitidos = producir(
    frames  = frames_source,
    producer= producer,
    fast    = False,      # True para modo turbo sin delays
    verbose = True,
)

print(f'\n✅ Producer finalizado: {n_emitidos} frames emitidos → {TOPIC}')

In [ ]:
# ── Modo threading: producer + consumer en la misma celda ─────────
# Útil en Colab donde no hay terminales separadas

import threading, time, os
from vocal_producer import conectar_producer, frames_desde_csv, frames_desde_perfil, producir
from vocal_consumer import consumir

RESULTADOS = {}

def run_consumer():
    global RESULTADOS
    time.sleep(2)  # Dar tiempo al producer para arrancar
    RESULTADOS = consumir(
        model_path = 'results/vocal_rf.pkl',
        out_path   = 'results/stream_output.json',
        verbose    = True,
    )

def run_producer():
    CSV_PATH = 'results/realtime_frames.csv'
    p = conectar_producer()
    src = frames_desde_csv(CSV_PATH) if os.path.exists(CSV_PATH) \
          else frames_desde_perfil('bogdan_style')
    producir(src, p, fast=True, verbose=True)  # fast=True para no esperar 90s

# Lanzar ambos en paralelo
t_consumer = threading.Thread(target=run_consumer, daemon=True)
t_producer = threading.Thread(target=run_producer)

print('🚀 Lanzando producer y consumer en paralelo...')
t_consumer.start()
t_producer.start()

t_producer.join()   # Esperar que el producer termine
t_consumer.join(timeout=15)  # Dar 15s al consumer para vaciar el buffer
print('\n✅ Pipeline completo. Resultados en RESULTADOS.')

## Sección 5 · Consumer: clasificación + anomalías

In [ ]:
# Importar el módulo del consumer
from vocal_consumer import consumir

# ── Consumer standalone (correr mientras el producer está activo) ─
# En Colab, usar el modo threading de la Sección 4.
# En terminal separada:
#   python vocal_consumer.py --model results/vocal_rf.pkl

print('Consumer configurado.')
print('Ejecutar desde terminal:')
print('  python vocal_consumer.py --model results/vocal_rf.pkl --out results/stream_output.json')
print()
print('O usar el modo threading de la Sección 4.')

In [ ]:
# ── Verificar tópicos en vivo (mientras producer corre) ───────────
import subprocess, time

KAFKA_DIR = '/content/kafka'

def lag_topico(topic, group='vocal-analyzer-v1'):
    """Muestra el offset y lag del consumer group."""
    r = subprocess.run(
        [f'{KAFKA_DIR}/bin/kafka-consumer-groups.sh',
         '--bootstrap-server','localhost:9092',
         '--describe','--group', group],
        capture_output=True, text=True, timeout=5
    )
    return r.stdout

def count_mensajes(topic):
    """Cuenta mensajes en un tópico."""
    r = subprocess.run(
        [f'{KAFKA_DIR}/bin/kafka-run-class.sh','kafka.tools.GetOffsetShell',
         '--broker-list','localhost:9092',
         '--topic', topic, '--time', '-1'],
        capture_output=True, text=True, timeout=5
    )
    try:
        return int(r.stdout.strip().split(':')[-1])
    except:
        return 0

# Snapshot del estado actual
print('=== Estado de tópicos ===')
for t in ['vocal.frames','vocal.analyzed','vocal.alerts']:
    n = count_mensajes(t)
    print(f'  {t:<22}: {n} mensajes')

## Sección 6 · Dashboard interactivo en vivo

In [ ]:
import json, os
from IPython.display import display, HTML

# Cargar resultados del stream
STREAM_JSON = 'results/stream_output.json'

if os.path.exists(STREAM_JSON):
    with open(STREAM_JSON) as f:
        stream_data = json.load(f)
    print(f'✅ Stream cargado: {len(stream_data["analyzed"])} ventanas analiz., '
          f'{len(stream_data["alerts"])} alertas')
else:
    print(f'⚠️  {STREAM_JSON} no encontrado. Usando datos demo.')
    # El dashboard HTML ya tiene datos embebidos

# Mostrar dashboard
DASHBOARD_PATH = 'kafka_dashboard.html'
if os.path.exists(DASHBOARD_PATH):
    with open(DASHBOARD_PATH, encoding='utf-8') as f:
        dashboard_html = f.read()
    display(HTML(dashboard_html))
else:
    print('Copiar kafka_dashboard.html al directorio de trabajo.')

## Sección 7 · Análisis de resultados del stream

In [ ]:
import json, os, pandas as pd
from collections import Counter

STREAM_JSON = 'results/stream_output.json'
if not os.path.exists(STREAM_JSON):
    print(f'No encontrado: {STREAM_JSON}')
else:
    with open(STREAM_JSON) as f:
        data = json.load(f)
    
    analyzed = data.get('analyzed', [])
    alerts   = data.get('alerts', [])
    
    if analyzed:
        df_a = pd.DataFrame(analyzed)
        print('=== VENTANAS ANALIZADAS ===')
        print(f'Total ventanas procesadas: {len(df_a)}')
        print(f'Predicciones únicas: {df_a["prediction"].nunique()}')
        print(f'Confianza promedio: {df_a["confidence"].mean():.1%}')
        print(f'Confianza mínima:   {df_a["confidence"].min():.1%}')
        print(f'Confianza máxima:   {df_a["confidence"].max():.1%}')
        print()
        print('Distribución de predicciones:')
        for pred, cnt in Counter(df_a['prediction']).most_common():
            pct = cnt/len(df_a)*100
            bar = '█' * int(pct/2)
            print(f'  {pred:<25} {cnt:>4}x  {pct:>5.1f}%  {bar}')
    
    if alerts:
        df_al = pd.DataFrame(alerts)
        print()
        print('=== ALERTAS EMITIDAS (vocal.alerts) ===')
        print(f'Total alertas: {len(df_al)}')
        for tipo, cnt in Counter(df_al['tipo']).most_common():
            icon = {'CLIMAX_VOCAL':'🔥','CAIDA_INTENS':'📉','AGUDO_EXTREMO':'🎯',
                    'SILENCIO_LARGO':'⏸','BREAK_VOZ':'⚡','INESTABILIDAD':'〰'}.get(tipo,'●')
            print(f'  {icon} {tipo:<20} {cnt}x')
        
        print()
        print('Alertas de severidad alta:')
        altas = df_al[df_al['severidad']=='alta']
        if len(altas) > 0:
            for _, row in altas.iterrows():
                print(f'  t={row["time"]:.1f}s  {row["tipo"]}  {row["descripcion"]}')
        else:
            print('  (sin alertas de severidad alta)')

In [ ]:
import matplotlib.pyplot as plt

if 'df_a' in dir() and len(df_a) > 0:
    plt.rcParams.update({'figure.facecolor':'#020b18','axes.facecolor':'#0a1628',
        'axes.edgecolor':'#0f2d4a','axes.labelcolor':'#94a3b8',
        'xtick.color':'#475569','ytick.color':'#475569','text.color':'#e2e8f0',
        'grid.color':'#0f2d4a'})
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Resultados del Stream Kafka Vocal', fontsize=12, fontweight='bold')
    
    # Confianza temporal
    axes[0].plot(df_a['t_start'], df_a['confidence'],
                 color='#38bdf8', linewidth=1.2, alpha=0.8)
    axes[0].fill_between(df_a['t_start'], df_a['confidence'],
                          alpha=0.15, color='#38bdf8')
    axes[0].set_xlabel('Tiempo (s)')
    axes[0].set_ylabel('Confianza')
    axes[0].set_title('Confianza del clasificador por ventana')
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, alpha=0.3)
    
    # Distribución predicciones
    counts = Counter(df_a['prediction'])
    COLORS = {'Barítono':'#a78bfa','Contratenor':'#34d399','Mezzo-soprano':'#fbbf24',
              'Soprano':'#f472b6','Tenor dramático':'#38bdf8','Tenor lírico-pop':'#fb923c'}
    labels = list(counts.keys())
    vals   = list(counts.values())
    colors = [COLORS.get(l,'#64748b') for l in labels]
    axes[1].bar(range(len(labels)), vals, color=colors, alpha=0.85)
    axes[1].set_xticks(range(len(labels)))
    axes[1].set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    axes[1].set_ylabel('Ventanas predichas')
    axes[1].set_title('Distribución de predicciones del stream')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('results/stream_analysis.png', dpi=150, bbox_inches='tight',
                facecolor='#020b18')
    plt.show()
    print('✅ Guardado: results/stream_analysis.png')

---

## 📋 Conclusiones

### Integración de los 4 trabajos

| Componente | Origen | Rol en este pipeline |
|------------|--------|----------------------|
| `realtime_frames.csv` | challenge_streaming (pipeline.py pYIN) | Fuente de datos del producer |
| `artist_profiles.py` | Idea C (comparador multi-artista) | Modo demo del producer |
| `VocalClassifier` | Idea A (clasificador ML) | Consumer: predice tipo vocal por ventana |
| `VocalAnomalyDetector` | Idea D (detección de anomalías) | Consumer: emite alertas a vocal.alerts |

### ¿Por qué Kafka aquí?

El pipeline original (`pipeline.py`) procesa el audio de forma **batch**: carga todo el archivo,
analiza todos los frames y devuelve los resultados al final. Kafka convierte ese mismo
análisis en **streaming real**: cada frame se emite en el momento en que sería detectado
por el sistema de análisis, el consumer clasifica ventanas a medida que llegan, y las
alertas aparecen en el dashboard exactamente cuando ocurren en el audio.

Esto habilita casos de uso que el pipeline original no soporta:
- Análisis de audio en vivo durante grabaciones
- Alertas en tiempo real para coaching vocal
- Múltiples consumers en paralelo (un consumer clasifica, otro guarda en base de datos)
- Replay del stream para debugging

### Diferencia clave respecto al trabajo de NLP

En el trabajo de reseñas gastronómicas, el producer emitía todos los mensajes lo más
rápido posible (batch simulado). Aquí el producer **respeta el timing del audio**:
un frame cada 50ms, exactamente la frecuencia de muestreo del análisis pYIN.
Esto hace que el stream sea temporal y que las métricas de ventana (2s = 40 frames)
correspondan exactamente a 2 segundos de audio real.